# Phase 2 — DAIC-WOZ Face Frame Extraction & Fine-tuning

**Thesis:** Multimodal Depression Risk Detection  
**Author:** Md. Mursalin, Netrokona University

This notebook covers Phase 2:
1. Load DAIC-WOZ videos and PHQ-8 depression labels
2. Extract face frames using OpenCV Haar cascade
3. Fine-tune the Phase 1 FER2013 CNN on binary depression classification
4. Evaluate with F1, AUC, confusion matrix

> **Note:** DAIC-WOZ requires approved access from https://dcapswoz.isi.edu/  
> Without it, the notebook runs a synthetic fallback to verify the pipeline.

## 1. Setup

In [ ]:
!git clone https://github.com/DevMursLab/Thesis.git depression_thesis
%cd depression_thesis
!pip install -q -r requirements.txt

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 2. DAIC-WOZ Data

**If you have DAIC-WOZ access**, upload the dataset to Google Drive and mount it:
```
data/raw/daic/
├── train_split_Depression_AVEC2017.csv
├── dev_split_Depression_AVEC2017.csv
├── 300_P/300_P.mp4
├── 301_P/301_P.mp4
└── ...
```
**Without access**, the script auto-generates synthetic data to verify the pipeline.

In [ ]:
# Optional: mount Google Drive if DAIC-WOZ is stored there
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/daic depression_thesis/data/raw/daic

import os
daic_path = 'data/raw/daic'
print('DAIC data found:', os.path.exists(daic_path))

## 3. Extract Face Frames

In [ ]:
!python src/preprocessing/daic_preprocess.py

In [ ]:
import numpy as np

X_train = np.load('data/processed/daic_faces/X_train.npy')
y_train = np.load('data/processed/daic_faces/y_train.npy')
X_dev   = np.load('data/processed/daic_faces/X_dev.npy')
y_dev   = np.load('data/processed/daic_faces/y_dev.npy')

print(f'Train: {X_train.shape}  |  Dev: {X_dev.shape}')
print(f'Train — Not depressed: {(y_train==0).sum()}  |  Depressed: {(y_train==1).sum()}')
print(f'Dev   — Not depressed: {(y_dev==0).sum()}    |  Depressed: {(y_dev==1).sum()}')

## 4. Visualize Extracted Frames

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
labels_text = {0: 'Not Depressed', 1: 'Depressed'}
colors = {0: 'green', 1: 'red'}

for row, cls in enumerate([0, 1]):
    idxs = np.where(y_train == cls)[0][:8]
    for col, i in enumerate(idxs):
        axes[row, col].imshow(X_train[i, 0], cmap='gray')
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(labels_text[cls], color=colors[cls], fontsize=9)

plt.suptitle('DAIC-WOZ — Extracted Face Frames', y=1.02)
plt.tight_layout()
plt.savefig('results/figures/daic_sample_frames.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Fine-tune Face CNN (Transfer from Phase 1)

If `results/face_cnn_phase1.pth` exists (Phase 1 completed), those weights are loaded automatically.

In [ ]:
!python src/training/train_face_daic.py

## 6. Detailed Evaluation

In [ ]:
import sys, torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_curve, auc)
sys.path.insert(0, '.')
from src.models.face_cnn import FaceCNN
from torch.utils.data import TensorDataset, DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = FaceCNN(num_classes=2).to(device)
model.load_state_dict(torch.load('results/face_cnn_phase2_daic.pth', map_location=device))
model.eval()

dev_ds = TensorDataset(torch.tensor(X_dev), torch.tensor(y_dev))
dev_dl = DataLoader(dev_ds, batch_size=64)

all_preds, all_probs, all_true = [], [], []
with torch.no_grad():
    for xb, yb in dev_dl:
        logits = model(xb.to(device))
        probs  = torch.softmax(logits, 1)[:, 1].cpu().numpy()
        preds  = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds); all_probs.extend(probs); all_true.extend(yb.numpy())

all_true  = np.array(all_true)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Confusion Matrix
cm = confusion_matrix(all_true, all_preds)
sns.heatmap(cm, annot=True, fmt='d', ax=ax1,
            xticklabels=['Not Dep', 'Depressed'],
            yticklabels=['Not Dep', 'Depressed'], cmap='Blues')
ax1.set_title('Confusion Matrix (Dev Set)'); ax1.set_ylabel('True'); ax1.set_xlabel('Predicted')

# ROC Curve
fpr, tpr, _ = roc_curve(all_true, all_probs)
roc_auc = auc(fpr, tpr)
ax2.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.3f}')
ax2.plot([0,1],[0,1],'k--'); ax2.set_xlabel('FPR'); ax2.set_ylabel('TPR')
ax2.set_title('ROC Curve — Face CNN (DAIC-WOZ)'); ax2.legend()

plt.tight_layout()
plt.savefig('results/figures/phase2_confusion_roc.png', dpi=150)
plt.show()

print(classification_report(all_true, all_preds,
                             target_names=['Not Depressed', 'Depressed']))

In [ ]:
import json
with open('results/metrics/phase2_metrics.json') as f:
    m = json.load(f)
print(json.dumps(m, indent=2))